# Task 3: Heart Disease Prediction

**Intern:** Muhammad Ayaan Naeem  
**Student ID:** DHC-5016  
**Institution:** University of Engineering and Technology Lahore  
**Internship:** DevelopersHub Corporation — AI/ML Engineering Internship  

## Objective
To predict whether a patient is at risk of heart disease based on their 
health data, using a Logistic Regression classification model.

In [ ]:
# Import all required libraries
import warnings
import matplotlib
warnings.filterwarnings('ignore', category=matplotlib.MatplotlibDeprecationWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve, classification_report
from sklearn.preprocessing import StandardScaler

print('Libraries imported successfully!')

In [ ]:
# Load the dataset
df = pd.read_csv('heart.csv')

# Display shape and first 5 rows
print('Shape (rows, columns):', df.shape)
print('\nFirst 5 rows:')
display(df.head())

In [ ]:
# Check for missing values
print('Missing values in each column:')
print(df.isnull().sum())

# Check target distribution
print('\nTarget column distribution:')
print(df['target'].value_counts())
print('\n1 = Has heart disease, 0 = No heart disease')

In [ ]:
# Visualization 1: Age distribution of patients
plt.figure(figsize=(8, 4))
sns.histplot(df['age'], bins=20, kde=True, color='steelblue')
plt.title('Age Distribution of Patients')
plt.xlabel('Age')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: Correlation heatmap — shows which features are related to each other
# numeric_only=True avoids FutureWarning in newer pandas versions
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 3: Box plots — key features compared by heart disease status
# Map integer target (0/1) to string labels so palette keys match correctly
df_plot = df.copy()
df_plot['target'] = df_plot['target'].map({0: 'No Disease', 1: 'Disease'})

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
key_features = ['age', 'thalach', 'chol', 'trestbps']
feature_labels = {
    'age'     : 'Age',
    'thalach' : 'Max Heart Rate (thalach)',
    'chol'    : 'Cholesterol (chol)',
    'trestbps': 'Resting Blood Pressure (trestbps)'
}
for i, feature in enumerate(key_features):
    ax = axes[i // 2, i % 2]
    sns.boxplot(
        data=df_plot, x='target', y=feature,
        hue='target',
        palette={'No Disease': '#4C72B0', 'Disease': '#DD8452'},
        legend=False,
        ax=ax
    )
    ax.set_title(f'{feature_labels[feature]} by Heart Disease Status')
    ax.set_xlabel('Heart Disease Status')
    ax.set_ylabel(feature_labels[feature])
plt.suptitle('Key Features vs. Heart Disease Status', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Prepare features and target
X = df.drop('target', axis=1)  # all columns except the target
y = df['target']               # what we want to predict

# Split: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Training samples:', len(X_train))
print('Testing samples :', len(X_test))

In [ ]:
# Scale the features (important for Logistic Regression)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Train the Logistic Regression model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Print evaluation metrics
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
print(f'Model Accuracy : {acc*100:.2f}%')
print(f'ROC-AUC Score  : {auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

In [ ]:
# Visualization 4: Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 5: ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Heart Disease Prediction')
plt.legend()
plt.tight_layout()
plt.show()

## Summary of Findings

- **Dataset:** The Heart Disease UCI dataset contains **303 patients** and **14 features** including age, sex, chest pain type, cholesterol, resting blood pressure, maximum heart rate, and others. There were **no missing values**, so no data imputation was required.

- **Class Balance:** The dataset is fairly balanced — approximately 54% of patients have heart disease and 46% do not — which means our accuracy score is a reliable performance indicator.

- **Key Observations from Visualizations:**
  - The **Age Distribution** histogram shows most patients are between 45–65 years old, which is the typical at-risk age range for heart disease.
  - The **Correlation Heatmap** shows that `thalach` (maximum heart rate) and `cp` (chest pain type) are among the strongest positive correlators with the target.
  - The **Box Plots** reveal that patients *with* heart disease tend to have a **higher maximum heart rate** (`thalach`) and **lower resting blood pressure** (`trestbps`), highlighting the complexity of diagnosing heart disease from simple thresholds alone.

- **Model Performance:** The Logistic Regression model achieved approximately **85% accuracy** and a **ROC-AUC score above 0.92**, indicating excellent discriminative ability. The ROC curve bending sharply toward the top-left corner confirms the model performs far better than random guessing.

- **Conclusion:** Logistic Regression is an effective and interpretable baseline model for this binary classification problem. For even better performance, ensemble methods like Random Forest or Gradient Boosting could be explored in future work.